### Colab Mount


In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    src_dir = Path("/content/drive/MyDrive/Research/AI-In-Science-Lab/local_learning/src")
else:
    cwd = Path.cwd().resolve()
    src_dir = cwd if (cwd / "models").is_dir() else cwd / "local_learning" / "src"

os.chdir(src_dir)
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))


### Setup


In [ ]:
import torch

from training import train

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {DEVICE}")


### Experiment Configs


In [ ]:
BASE_CONFIG = {
    "project": "local-learning",
    "dataset": "cifar10",
    "preprocessing": "none",
    "training_mode": "classification",
    "loss_type": "regular",
    "learning_rate": 1e-3,
    "epochs": 3,
    "batch_size": 64,
    "num_workers": 2,
    "lambda_strength": 0.0,
    "kernel_size": 3,
    "correlation_mode": "max",
    "comparison_mode": "shift",
    "normalization_mode": "relu",
    "postcomp_mode": "squared",
    "correlation_loss": "sum",
    "local": False,
    "local_training": False,
    "local_reconstruction_strength": 1.0,
    "hidden_ch": 64,
    "hidden_channels": [64, 128],
    "num_layers": 2,
    "k_spatial": 0.2,
    "k_population": None,
    "k_lifetime": 0.2,
    "wta_eval_multiplier": 1.0,
    "log_every_steps": 100,
    "log_weight_grids_every_steps": 100,
    "log_flops_every_steps": 0,
    "max_eval_batches": 25,
}

BASIC_CNN_CONFIG = {
    **BASE_CONFIG,
    "architecture_type": "cnn1",
}

GREEDY_POPULATION_CONFIG = {
    **BASE_CONFIG,
    "architecture_type": "greedy_stacked_autoencoder",
    "k_population": 0.2,
    "k_lifetime": None,
    "local_training": False,
}

GREEDY_POPULATION_LOCAL_CONFIG = {
    **GREEDY_POPULATION_CONFIG,
    "local_training": True,
    "local_reconstruction_strength": 1.0,
}

EXPERIMENT_CONFIGS = {
    "basic_cnn": BASIC_CNN_CONFIG,
    "greedy_population": GREEDY_POPULATION_CONFIG,
    "greedy_population_local": GREEDY_POPULATION_LOCAL_CONFIG,
}


### Run


In [ ]:
SELECTED_CONFIG = "basic_cnn"
# SELECTED_CONFIG = "greedy_population"
# SELECTED_CONFIG = "greedy_population_local"

metrics = train(EXPERIMENT_CONFIGS[SELECTED_CONFIG], device=DEVICE)
metrics
